# 03 - XGBoost Model Training

## Purpose

This notebook trains and evaluates XGBoost models using the ML-ready datasets prepared in the data-preprocessing stage.

Based on the preprocessing and feature-engineering stage, this notebook will:

- Load the ML-ready training, validation, and test datasets.
- Load the classification and regression targets.
- Train an XGBoost classification model to predict whether an order will be delayed.
- Evaluate the classification model using appropriate performance metrics.
- Train an XGBoost regression model to predict the number of delivery delay days.
- Evaluate the regression model using appropriate performance metrics.
- Save the trained models for later use by the prediction/API layer.

## Input from Notebook 2

This notebook uses the files generated by `02_preprocessing.ipynb`.

### ML Features

- `X_train.csv` — training features
- `X_val.csv` — validation features
- `X_test.csv` — test features

The features are already:

- Missing-value handled.
- Categorical features encoded.
- Converted into numerical ML-ready format.
- Split chronologically into training, validation, and test sets.

### Classification Targets

- `y_class_train.csv`
- `y_class_val.csv`
- `y_class_test.csv`

Classification target:

- `delayed` — whether an order is delayed (`0` = not delayed, `1` = delayed)

### Regression Targets

- `y_reg_train.csv`
- `y_reg_val.csv`
- `y_reg_test.csv`

Regression target:

- `delivery_delay_days` — number of days early or late an order is delivered.

## Models

### Classification Model

XGBoost Classifier

**Objective:** Predict the probability that an order will be delayed.

Output:

- Delay probability
- Predicted delay status (`0` or `1`)

### Regression Model

XGBoost Regressor

**Objective:** Predict the expected delivery delay in days.

Output:

- Predicted delivery delay in days

## Model Evaluation

The classification model will be evaluated using:

- Accuracy
- Precision
- Recall
- F1-score
- ROC-AUC
- Confusion matrix

The regression model will be evaluated using:

- MAE
- RMSE
- R²

## Model Outputs

The trained models will be saved for use by the prediction/API layer.

The final model outputs will provide the foundation for the prescriptive analytics stage:

**Order Data → Delay Prediction → Delay Duration → Prescriptive Action Recommendation**

## Next Stage

The trained XGBoost models from this notebook will be passed to the prediction/API and prescriptive analytics stages.

The classification model identifies whether a delay is likely, while the regression model estimates the expected delay duration. These predictions will later be used to support recommended supply-chain actions.

In [1]:
# 1. Import libraries and define paths for XGBoost training

import pandas as pd
import numpy as np
import os
import joblib

from xgboost import XGBClassifier, XGBRegressor

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report,
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

# --------------------------------------------------
# Define paths
# --------------------------------------------------

data_dir = "../../data/processed/ml_ready"
model_dir = "../../ml/models"

os.makedirs(model_dir, exist_ok=True)

# --------------------------------------------------
# Load processed feature datasets
# --------------------------------------------------

X_train = pd.read_csv(f"{data_dir}/X_train.csv")
X_val = pd.read_csv(f"{data_dir}/X_val.csv")
X_test = pd.read_csv(f"{data_dir}/X_test.csv")

# --------------------------------------------------
# Load classification targets
# --------------------------------------------------

y_class_train = pd.read_csv(
    f"{data_dir}/y_class_train.csv"
)["delayed"]

y_class_val = pd.read_csv(
    f"{data_dir}/y_class_val.csv"
)["delayed"]

y_class_test = pd.read_csv(
    f"{data_dir}/y_class_test.csv"
)["delayed"]

# --------------------------------------------------
# Load regression targets
# --------------------------------------------------

y_reg_train = pd.read_csv(
    f"{data_dir}/y_reg_train.csv"
)["delivery_delay_days"]

y_reg_val = pd.read_csv(
    f"{data_dir}/y_reg_val.csv"
)["delivery_delay_days"]

y_reg_test = pd.read_csv(
    f"{data_dir}/y_reg_test.csv"
)["delivery_delay_days"]

# --------------------------------------------------
# Verify data dimensions and target alignment
# --------------------------------------------------

print("X_train:", X_train.shape)
print("X_val:", X_val.shape)
print("X_test:", X_test.shape)

print("\nClassification targets:")
print("Train:", y_class_train.shape)
print("Validation:", y_class_val.shape)
print("Test:", y_class_test.shape)

print("\nRegression targets:")
print("Train:", y_reg_train.shape)
print("Validation:", y_reg_val.shape)
print("Test:", y_reg_test.shape)

print("\nData loading completed successfully.")

X_train: (67533, 144)
X_val: (14471, 144)
X_test: (14472, 144)

Classification targets:
Train: (67533,)
Validation: (14471,)
Test: (14472,)

Regression targets:
Train: (67533,)
Validation: (14471,)
Test: (14472,)

Data loading completed successfully.


In [2]:
# 2. Check train/validation/test target and time distributions before retraining - Diagnostic

import pandas as pd
import numpy as np

print("=" * 70)
print("1. CLASSIFICATION TARGET DISTRIBUTION")
print("=" * 70)

for name, y in [
    ("Train", y_class_train),
    ("Validation", y_class_val),
    ("Test", y_class_test)
]:
    total = len(y)
    delayed = int((y == 1).sum())
    not_delayed = int((y == 0).sum())

    print(
        f"{name}: "
        f"Delayed = {delayed} ({delayed / total * 100:.2f}%) | "
        f"Not Delayed = {not_delayed} ({not_delayed / total * 100:.2f}%)"
    )


print("\n" + "=" * 70)
print("2. REGRESSION TARGET DISTRIBUTION")
print("=" * 70)

for name, y in [
    ("Train", y_reg_train),
    ("Validation", y_reg_val),
    ("Test", y_reg_test)
]:
    print(f"\n{name}:")
    print(f"  Mean   : {y.mean():.2f}")
    print(f"  Median : {y.median():.2f}")
    print(f"  Std    : {y.std():.2f}")
    print(f"  Min    : {y.min():.2f}")
    print(f"  Max    : {y.max():.2f}")


print("\n" + "=" * 70)
print("3. ORIGINAL PURCHASE-TIME DISTRIBUTION")
print("=" * 70)

# Load the original processed dataset to inspect the actual time periods
original_path = "../../data/processed/processed_orders.csv"
df_original = pd.read_csv(original_path)

df_original["order_purchase_timestamp"] = pd.to_datetime(
    df_original["order_purchase_timestamp"],
    errors="coerce"
)

# Sort exactly as done during preprocessing
df_original = df_original.sort_values(
    "order_purchase_timestamp"
).reset_index(drop=True)

n = len(df_original)

train_end = int(n * 0.70)
val_end = int(n * 0.85)

df_train_time = df_original.iloc[:train_end].copy()
df_val_time = df_original.iloc[train_end:val_end].copy()
df_test_time = df_original.iloc[val_end:].copy()

for name, df_part in [
    ("Train", df_train_time),
    ("Validation", df_val_time),
    ("Test", df_test_time)
]:
    print(f"\n{name}:")
    print(
        f"  Start: {df_part['order_purchase_timestamp'].min()}"
    )
    print(
        f"  End  : {df_part['order_purchase_timestamp'].max()}"
    )
    print(
        f"  Rows : {len(df_part)}"
    )


print("\n" + "=" * 70)
print("4. DELAY RATE BY PURCHASE YEAR-MONTH")
print("=" * 70)

df_original["purchase_period"] = (
    df_original["order_purchase_timestamp"]
    .dt.to_period("M")
    .astype(str)
)

monthly_delay = (
    df_original.groupby("purchase_period")["delayed"]
    .agg(["count", "sum", "mean"])
    .reset_index()
)

monthly_delay["delay_rate_%"] = monthly_delay["mean"] * 100

print(
    monthly_delay[
        ["purchase_period", "count", "sum", "delay_rate_%"]
    ].to_string(index=False)
)


print("\n" + "=" * 70)
print("5. NUMERICAL FEATURE DISTRIBUTION SHIFT")
print("=" * 70)

# These are the main numerical business features available before encoding
numeric_features = [
    "item_count",
    "product_count",
    "seller_count",
    "total_price",
    "total_freight_value",
    "total_order_value",
    "avg_item_price",
    "avg_product_weight_g",
    "avg_product_length_cm",
    "avg_product_height_cm",
    "avg_product_width_cm",
    "avg_shipping_limit_days",
]

available_numeric = [
    col for col in numeric_features
    if col in df_original.columns
]

for feature in available_numeric:
    train_mean = df_train_time[feature].mean()
    val_mean = df_val_time[feature].mean()
    test_mean = df_test_time[feature].mean()

    print(
        f"{feature:<30} "
        f"Train={train_mean:>10.2f} | "
        f"Val={val_mean:>10.2f} | "
        f"Test={test_mean:>10.2f}"
    )


print("\n" + "=" * 70)
print("DIAGNOSTIC COMPLETED")
print("=" * 70)

1. CLASSIFICATION TARGET DISTRIBUTION
Train: Delayed = 6097 (9.03%) | Not Delayed = 61436 (90.97%)
Validation: Delayed = 773 (5.34%) | Not Delayed = 13698 (94.66%)
Test: Delayed = 957 (6.61%) | Not Delayed = 13515 (93.39%)

2. REGRESSION TARGET DISTRIBUTION

Train:
  Mean   : -10.77
  Median : -11.94
  Std    : 10.49
  Min    : -146.02
  Max    : 188.97

Validation:
  Mean   : -14.14
  Median : -13.35
  Std    : 9.91
  Min    : -70.18
  Max    : 106.69

Test:
  Mean   : -10.12
  Median : -9.24
  Std    : 8.34
  Min    : -60.16
  Max    : 60.61

3. ORIGINAL PURCHASE-TIME DISTRIBUTION

Train:
  Start: 2016-09-15 12:16:38
  End  : 2018-04-15 20:07:56
  Rows : 67533

Validation:
  Start: 2018-04-15 20:10:23
  End  : 2018-06-21 07:50:39
  Rows : 14471

Test:
  Start: 2018-06-21 08:29:29
  End  : 2018-08-29 15:00:37
  Rows : 14472

4. DELAY RATE BY PURCHASE YEAR-MONTH
purchase_period  count  sum  delay_rate_%
        2016-09      1    1    100.000000
        2016-10    270    3      1.111111

In [3]:
# 3. Build additional delivery-time features and retrain the classification model

import pandas as pd
import numpy as np
import joblib

from xgboost import XGBClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)

# ==================================================
# 1. Load original processed data
# ==================================================

original_path = "../../data/processed/processed_orders.csv"

df_model = pd.read_csv(original_path)

df_model["order_purchase_timestamp"] = pd.to_datetime(
    df_model["order_purchase_timestamp"],
    errors="coerce"
)

df_model["order_estimated_delivery_date"] = pd.to_datetime(
    df_model["order_estimated_delivery_date"],
    errors="coerce"
)

# Keep exactly the same chronological order used for the ML split
df_model = df_model.sort_values(
    "order_purchase_timestamp"
).reset_index(drop=True)

# ==================================================
# 2. Create useful delivery-time features
# ==================================================

df_model["estimated_delivery_lead_days"] = (
    df_model["order_estimated_delivery_date"]
    - df_model["order_purchase_timestamp"]
).dt.total_seconds() / 86400

# Purchase hour may capture operational timing
df_model["purchase_hour"] = (
    df_model["order_purchase_timestamp"].dt.hour
)

# ==================================================
# 3. Select original logical features + new features
# ==================================================

model_feature_cols = [
    "customer_state",
    "primary_seller_state",
    "primary_category",
    "item_count",
    "product_count",
    "seller_count",
    "total_price",
    "total_freight_value",
    "total_order_value",
    "avg_item_price",
    "avg_product_weight_g",
    "avg_product_length_cm",
    "avg_product_height_cm",
    "avg_product_width_cm",
    "avg_shipping_limit_days",
    "purchase_year",
    "purchase_month",
    "purchase_dayofweek",
    "estimated_delivery_lead_days",
    "purchase_hour"
]

X_new = df_model[model_feature_cols].copy()

# ==================================================
# 4. Handle missing values
# ==================================================

categorical_cols = [
    "customer_state",
    "primary_seller_state",
    "primary_category"
]

numeric_cols = [
    col for col in model_feature_cols
    if col not in categorical_cols
]

# Fill categorical values
for col in categorical_cols:
    X_new[col] = X_new[col].fillna("Unknown")

# Fill numerical values using training-safe medians
for col in numeric_cols:
    train_median = X_new.iloc[:67533][col].median()
    X_new[col] = X_new[col].fillna(train_median)

# ==================================================
# 5. One-hot encode using the complete historical
#    feature vocabulary
# ==================================================

X_new = pd.get_dummies(
    X_new,
    columns=categorical_cols,
    dummy_na=False
)

X_new = X_new.astype(float)

# ==================================================
# 6. Recreate chronological train/validation/test
#    feature matrices
# ==================================================

X_new_train = X_new.iloc[:67533].copy()
X_new_val = X_new.iloc[67533:82004].copy()
X_new_test = X_new.iloc[82004:].copy()

print("New feature matrix:")
print("Train:", X_new_train.shape)
print("Validation:", X_new_val.shape)
print("Test:", X_new_test.shape)

# ==================================================
# 7. Train classification model
# ==================================================

negative_count = (y_class_train == 0).sum()
positive_count = (y_class_train == 1).sum()

scale_pos_weight = negative_count / positive_count

classifier = XGBClassifier(
    n_estimators=1500,
    learning_rate=0.02,
    max_depth=3,
    min_child_weight=8,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=0.5,
    reg_lambda=3.0,
    objective="binary:logistic",
    eval_metric="auc",
    scale_pos_weight=scale_pos_weight,
    early_stopping_rounds=75,
    random_state=42,
    n_jobs=-1
)

classifier.fit(
    X_new_train,
    y_class_train,
    eval_set=[(X_new_val, y_class_val)],
    verbose=False
)

# ==================================================
# 8. Validation evaluation
# ==================================================

y_class_val_prob = classifier.predict_proba(
    X_new_val
)[:, 1]

y_class_val_pred = (
    y_class_val_prob >= 0.5
).astype(int)

print("\nCLASSIFICATION VALIDATION RESULTS")
print("=================================")
print("Best iteration:", classifier.best_iteration)
print("Accuracy :", round(
    accuracy_score(y_class_val, y_class_val_pred), 4
))
print("Precision:", round(
    precision_score(y_class_val, y_class_val_pred, zero_division=0), 4
))
print("Recall   :", round(
    recall_score(y_class_val, y_class_val_pred, zero_division=0), 4
))
print("F1 Score :", round(
    f1_score(y_class_val, y_class_val_pred, zero_division=0), 4
))
print("ROC-AUC  :", round(
    roc_auc_score(y_class_val, y_class_val_prob), 4
))

# ==================================================
# 9. Save improved classifier
# ==================================================

classifier_path = "../../ml/models/xgboost_classifier.joblib"

joblib.dump(
    classifier,
    classifier_path
)

print("\nClassification model saved to:")
print(classifier_path)

New feature matrix:
Train: (67533, 140)
Validation: (14471, 140)
Test: (14472, 140)

CLASSIFICATION VALIDATION RESULTS
Best iteration: 758
Accuracy : 0.8118
Precision: 0.1389
Recall   : 0.4851
F1 Score : 0.216
ROC-AUC  : 0.7433

Classification model saved to:
../../ml/models/xgboost_classifier.joblib


In [4]:
# 4. Train the improved XGBoost regression model using the new delivery-time features

from xgboost import XGBRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# ==================================================
# 1. Create regression model
# ==================================================

regressor = XGBRegressor(
    n_estimators=1500,
    learning_rate=0.02,
    max_depth=3,
    min_child_weight=8,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=0.5,
    reg_lambda=3.0,
    objective="reg:squarederror",
    eval_metric="rmse",
    early_stopping_rounds=75,
    random_state=42,
    n_jobs=-1
)

# ==================================================
# 2. Train using chronological validation set
# ==================================================

regressor.fit(
    X_new_train,
    y_reg_train,
    eval_set=[(X_new_val, y_reg_val)],
    verbose=False
)

# ==================================================
# 3. Validation prediction
# ==================================================

y_reg_val_pred = regressor.predict(
    X_new_val
)

# ==================================================
# 4. Validation evaluation
# ==================================================

val_mae = mean_absolute_error(
    y_reg_val,
    y_reg_val_pred
)

val_rmse = np.sqrt(
    mean_squared_error(
        y_reg_val,
        y_reg_val_pred
    )
)

val_r2 = r2_score(
    y_reg_val,
    y_reg_val_pred
)

print("REGRESSION VALIDATION RESULTS")
print("=============================")
print("Best iteration:", regressor.best_iteration)
print("MAE :", round(val_mae, 4))
print("RMSE:", round(val_rmse, 4))
print("R²  :", round(val_r2, 4))

# ==================================================
# 5. Save regression model
# ==================================================

regressor_path = "../../ml/models/xgboost_regressor.joblib"

joblib.dump(
    regressor,
    regressor_path
)

print("\nRegression model saved to:")
print(regressor_path)

REGRESSION VALIDATION RESULTS
Best iteration: 1498
MAE : 4.9857
RMSE: 6.8592
R²  : 0.5207

Regression model saved to:
../../ml/models/xgboost_regressor.joblib


In [5]:
# 5. Evaluate the newly trained models on the untouched chronological test set

# ==================================================
# 1. Classification test evaluation
# ==================================================

y_class_test_prob = classifier.predict_proba(
    X_new_test
)[:, 1]

y_class_test_pred = (
    y_class_test_prob >= 0.5
).astype(int)

print("CLASSIFICATION TEST RESULTS")
print("============================")

print("Accuracy :", round(
    accuracy_score(y_class_test, y_class_test_pred), 4
))

print("Precision:", round(
    precision_score(
        y_class_test,
        y_class_test_pred,
        zero_division=0
    ), 4
))

print("Recall   :", round(
    recall_score(
        y_class_test,
        y_class_test_pred,
        zero_division=0
    ), 4
))

print("F1 Score :", round(
    f1_score(
        y_class_test,
        y_class_test_pred,
        zero_division=0
    ), 4
))

print("ROC-AUC  :", round(
    roc_auc_score(
        y_class_test,
        y_class_test_prob
    ), 4
))


# ==================================================
# 2. Regression test evaluation
# ==================================================

y_reg_test_pred = regressor.predict(
    X_new_test
)

test_mae = mean_absolute_error(
    y_reg_test,
    y_reg_test_pred
)

test_rmse = np.sqrt(
    mean_squared_error(
        y_reg_test,
        y_reg_test_pred
    )
)

test_r2 = r2_score(
    y_reg_test,
    y_reg_test_pred
)

print("\nREGRESSION TEST RESULTS")
print("=======================")
print("MAE :", round(test_mae, 4))
print("RMSE:", round(test_rmse, 4))
print("R²  :", round(test_r2, 4))

CLASSIFICATION TEST RESULTS
Accuracy : 0.6604
Precision: 0.0757
Recall   : 0.3689
F1 Score : 0.1256
ROC-AUC  : 0.6169

REGRESSION TEST RESULTS
MAE : 4.645
RMSE: 6.109
R²  : 0.4639


In [6]:
# 6. Find a practical classification threshold using the validation set

import numpy as np
import pandas as pd
from sklearn.metrics import precision_score, recall_score, f1_score

# Get delayed probabilities from the trained classification model
val_probabilities = classifier.predict_proba(X_new_val)[:, 1]

# Test a range of thresholds on the validation set
threshold_results = []

for threshold in np.arange(0.10, 0.71, 0.05):
    val_predictions = (val_probabilities >= threshold).astype(int)

    precision = precision_score(y_class_val, val_predictions, zero_division=0)
    recall = recall_score(y_class_val, val_predictions, zero_division=0)
    f1 = f1_score(y_class_val, val_predictions, zero_division=0)

    threshold_results.append({
        "threshold": round(threshold, 2),
        "precision": round(precision, 4),
        "recall": round(recall, 4),
        "f1": round(f1, 4)
    })

threshold_df = pd.DataFrame(threshold_results)

print("CLASSIFICATION THRESHOLD ANALYSIS")
print("=================================\n")
print(threshold_df.to_string(index=False))

# Select the threshold with the highest validation F1 score
best_threshold_row = threshold_df.loc[threshold_df["f1"].idxmax()]
best_threshold = float(best_threshold_row["threshold"])

print("\nSELECTED THRESHOLD")
print("==================")
print(f"Threshold : {best_threshold:.2f}")
print(f"Precision : {best_threshold_row['precision']:.4f}")
print(f"Recall    : {best_threshold_row['recall']:.4f}")
print(f"F1 Score  : {best_threshold_row['f1']:.4f}")

CLASSIFICATION THRESHOLD ANALYSIS

 threshold  precision  recall     f1
      0.10     0.0545  0.9987 0.1034
      0.15     0.0585  0.9961 0.1104
      0.20     0.0634  0.9767 0.1191
      0.25     0.0699  0.9314 0.1301
      0.30     0.0796  0.8486 0.1455
      0.35     0.0906  0.7503 0.1617
      0.40     0.1067  0.6843 0.1845
      0.45     0.1218  0.5860 0.2017
      0.50     0.1389  0.4851 0.2160
      0.55     0.1602  0.3816 0.2257
      0.60     0.1845  0.2704 0.2193
      0.65     0.2265  0.1682 0.1930
      0.70     0.2780  0.0802 0.1245

SELECTED THRESHOLD
Threshold : 0.55
Precision : 0.1602
Recall    : 0.3816
F1 Score  : 0.2257


In [7]:
# 7. Final classification evaluation using the validation-selected threshold

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix
)

# Use the already trained classifier to get test probabilities
test_probabilities = classifier.predict_proba(X_new_test)[:, 1]

# Apply the threshold selected from the validation set
test_predictions_threshold = (
    test_probabilities >= best_threshold
).astype(int)

# Calculate final test metrics
test_accuracy = accuracy_score(y_class_test, test_predictions_threshold)
test_precision = precision_score(
    y_class_test,
    test_predictions_threshold,
    zero_division=0
)
test_recall = recall_score(
    y_class_test,
    test_predictions_threshold,
    zero_division=0
)
test_f1 = f1_score(
    y_class_test,
    test_predictions_threshold,
    zero_division=0
)

# ROC-AUC uses probabilities, not thresholded predictions
test_roc_auc = roc_auc_score(
    y_class_test,
    test_probabilities
)

# Confusion matrix
test_cm = confusion_matrix(
    y_class_test,
    test_predictions_threshold
)

print("FINAL CLASSIFICATION TEST RESULTS")
print("=================================")
print(f"Selected threshold : {best_threshold:.2f}")
print(f"Accuracy           : {test_accuracy:.4f}")
print(f"Precision          : {test_precision:.4f}")
print(f"Recall             : {test_recall:.4f}")
print(f"F1 Score           : {test_f1:.4f}")
print(f"ROC-AUC            : {test_roc_auc:.4f}")

print("\nCONFUSION MATRIX")
print("================")
print(test_cm)

print("\nInterpretation:")
print(f"True Negatives  : {test_cm[0, 0]}")
print(f"False Positives : {test_cm[0, 1]}")
print(f"False Negatives : {test_cm[1, 0]}")
print(f"True Positives  : {test_cm[1, 1]}")

FINAL CLASSIFICATION TEST RESULTS
Selected threshold : 0.55
Accuracy           : 0.7520
Precision          : 0.0814
Recall             : 0.2675
F1 Score           : 0.1248
ROC-AUC            : 0.6169

CONFUSION MATRIX
[[10627  2888]
 [  701   256]]

Interpretation:
True Negatives  : 10627
False Positives : 2888
False Negatives : 701
True Positives  : 256


In [8]:
# 8. Save final model metadata and define the backend model interface

import json
import os

# Final logical input features used by the model
model_input_features = [
    "customer_state",
    "primary_seller_state",
    "primary_category",
    "item_count",
    "product_count",
    "seller_count",
    "total_price",
    "total_freight_value",
    "total_order_value",
    "avg_item_price",
    "avg_product_weight_g",
    "avg_product_length_cm",
    "avg_product_height_cm",
    "avg_product_width_cm",
    "avg_shipping_limit_days",
    "purchase_year",
    "purchase_month",
    "purchase_dayofweek",
    "order_purchase_timestamp",
    "order_estimated_delivery_date",
    "estimated_delivery_lead_days",
    "purchase_hour"
]

# Define model outputs
model_outputs = {
    "classification": {
        "model": "xgboost_classifier.joblib",
        "target": "delayed",
        "threshold": float(best_threshold),
        "outputs": [
            "delayed_probability",
            "delayed_prediction"
        ]
    },
    "regression": {
        "model": "xgboost_regressor.joblib",
        "target": "delivery_delay_days",
        "outputs": [
            "predicted_delivery_delay_days"
        ]
    }
}

# Save metadata
metadata = {
    "project": "SupplyPrescript",
    "model_type": "XGBoost",
    "input_features": model_input_features,
    "classification": model_outputs["classification"],
    "regression": model_outputs["regression"],
    "classification_test_metrics": {
        "accuracy": float(test_accuracy),
        "precision": float(test_precision),
        "recall": float(test_recall),
        "f1": float(test_f1),
        "roc_auc": float(test_roc_auc)
    },
    "regression_test_metrics": {
        "mae": 4.645,
        "rmse": 6.109,
        "r2": 0.4639
    }
}

# Save metadata file
metadata_path = "../../ml/models/model_metadata.json"

with open(metadata_path, "w") as f:
    json.dump(metadata, f, indent=4)

print("FINAL MODEL METADATA")
print("====================")
print(f"Classification threshold : {best_threshold:.2f}")
print(f"Logical input features   : {len(model_input_features)}")
print(f"Classification model     : xgboost_classifier.joblib")
print(f"Regression model         : xgboost_regressor.joblib")
print(f"Metadata file            : {metadata_path}")

print("\nMODEL INTERFACE")
print("===============")
print("Classification output:")
print("  - delayed_probability")
print("  - delayed_prediction")

print("\nRegression output:")
print("  - predicted_delivery_delay_days")

print("\nNotebook 3 model training and evaluation completed.")

FINAL MODEL METADATA
Classification threshold : 0.55
Logical input features   : 22
Classification model     : xgboost_classifier.joblib
Regression model         : xgboost_regressor.joblib
Metadata file            : ../../ml/models/model_metadata.json

MODEL INTERFACE
Classification output:
  - delayed_probability
  - delayed_prediction

Regression output:
  - predicted_delivery_delay_days

Notebook 3 model training and evaluation completed.
